# CIFAR-10 – konvoluutioneuroverkko (CNN)

## Johdanto

Tässä notebookissa CIFAR-10-kuvia luokitellaan konvoluutioneuroverkolla (CNN), ja tuloksia verrataan FCN-notebookin tuloksiin. Pohjana on käytetty MNIST-datasetille tehtyä Keras CNN -työkirjaa, jota on muokattu kolmikanavaisille värikuville ja syvemmäksi.

Malleja on kolme, ja jokainen lisää yhden parannuksen edelliseen:
1. perus-CNN (kaksi konvoluutiokerrosta, kuten MNIST-mallissa)
2. syvempi CNN, jossa Batch Normalization ja Dropout
3. sama malli + data-augmentaatio

Näin jokaisen tekniikan vaikutus näkyy erikseen. Lopuksi tarkastellaan, mitä ensimmäinen konvoluutiokerros oppii, ja verrataan CNN:ää ja FCN:ää. **FCN-notebook pitää ajaa ensin**, koska vertailu lukee sen tulokset tiedostosta `results_fcn.json`.

Tehtävänannon tavoitteet CNN:lle: realistinen 75–80 %, kunnianhimoinen 82–85 % testitarkkuus.

In [ ]:
EPOCHS = 60
PATIENCE = 10

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time, json
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

keras.utils.set_random_seed(42)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## Data

CIFAR-10 sisältää 60 000 värikuvaa (32×32 pikseliä, 3 värikanavaa) kymmenestä luokasta: lentokone, auto, lintu, kissa, peura, koira, sammakko, hevonen, laiva ja kuorma-auto. Opetusdataa on 50 000 ja testidataa 10 000 kuvaa, ja luokat ovat tasakokoisia (5 000 / 1 000 kuvaa luokkaa kohden), joten tarkkuus on järkevä päämittari.

Verrattuna MNISTiin data on selvästi vaikeampaa: kohde voi olla missä kohtaa kuvaa tahansa, eri kokoisena ja eri kulmasta, taustat vaihtelevat, ja osa luokista muistuttaa toisiaan (kissa/koira, auto/kuorma-auto, peura/hevonen). Pieni resoluutio vaikeuttaa tunnistamista myös ihmiselle. Alla on 20 ensimmäistä opetuskuvaa nimikkeineen.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("train:", x_train.shape, "test:", x_test.shape)

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]], fontsize=7)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Esikäsittely

Pikseliarvot muutetaan ensin liukuluvuiksi ja skaalataan välille 0–1. Sen jälkeen jokainen värikanava (R, G, B) standardoidaan erikseen: kanavasta vähennetään sen keskiarvo ja jaetaan keskihajonnalla. Kanavat käsitellään erikseen, koska niiden jakaumat poikkeavat toisistaan (esim. sininen kanava on keskimäärin eri tasolla kuin punainen). Standardoitu syöte on keskitetty nollan ympärille, mikä tasoittaa gradientteja ja nopeuttaa ja vakauttaa oppimista.

Keskiarvo ja hajonta lasketaan vain opetusdatasta ja samoja arvoja käytetään testidatalle. Näin testidatasta ei vuoda tietoa malliin, ja testitulos kuvaa aidosti suoriutumista ennennäkemättömällä datalla. Värikuvan muoto (32, 32, 3) säilyy esikäsittelyssä ennallaan.

Esikäsittely on täsmälleen sama kuin FCN-notebookissa, jotta mallien erot johtuvat arkkitehtuurista eivätkä datan käsittelystä.

In [ ]:
# Skaalaus 0-1 ja kanavakohtainen standardointi (keskiarvo ja hajonta lasketaan vain opetusdatasta)
x_train_f = x_train.astype("float32") / 255.0
x_test_f = x_test.astype("float32") / 255.0

mean = x_train_f.mean(axis=(0, 1, 2))
std = x_train_f.std(axis=(0, 1, 2))
print("kanavien keskiarvot (R,G,B):", mean.round(3), "hajonnat:", std.round(3))

x_train_n = (x_train_f - mean) / std
x_test_n = (x_test_f - mean) / std

## Data-augmentaatio

Data-augmentaatiossa opetuskuvia muunnellaan satunnaisesti jokaisella epookilla, jolloin malli ei näe täsmälleen samaa kuvaa kahdesti. Käytössä on kaksi muunnosta:
- **RandomFlip("horizontal")** peilaa kuvan vaakasuunnassa 50 %:n todennäköisyydellä. Peilattu kissa on yhä kissa, joten luokka ei muutu. Pystysuuntaista peilausta ei käytetä, koska ylösalainen auto tai laiva ei ole luonteva kuva.
- **RandomTranslation(0.1, 0.1)** siirtää kuvaa enintään 10 % (noin 3 pikseliä) kumpaankin suuntaan. Reunat täytetään peilaamalla (`fill_mode="reflect"`), jotta kuvaan ei tule mustia raitoja.

Muunnokset opettavat mallille, että kohteen sijainti tai suunta ei vaikuta luokkaan, ja ne toimivat tehokkaana regularisointina: opetusdata on käytännössä paljon suurempi kuin 50 000 kuvaa.

Augmentaatio on toteutettu mallin sisäisenä kerroksena, ja Kerasin esikäsittelykerrokset ovat aktiivisia vain koulutuksessa (`training=True`). Validointi- ja testivaiheessa kuvat menevät malliin muuttumattomina, koska tarkoitus on arvioida mallia oikeilla kuvilla eikä satunnaisesti muunnelluilla. Alla on saman kuvan kahdeksan satunnaista muunnosta.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
], name="augment")

plt.figure(figsize=(10, 2))
img = x_train_n[:1]
for i in range(8):
    plt.subplot(1, 8, i + 1)
    out = augment(img, training=True)[0].numpy() * std + mean
    plt.imshow(np.clip(out, 0, 1)); plt.axis("off")
plt.suptitle("Saman kuvan augmentoituja versioita"); plt.show()

## Mallit

**Perus-CNN (`cnn_baseline`)**: kaksi Conv2D (32 ja 64 suodatinta, 3×3) + MaxPooling -paria, minkä jälkeen Flatten, Dense 64 ja softmax. Rakenne vastaa MNIST-mallia.

**Parannettu CNN (`cnn_bn_dropout`)**: kolme konvoluutiolohkoa (32 → 64 → 128 suodatinta). Jokaisessa lohkossa on kaksi 3×3-konvoluutiota peräkkäin (VGG-tyylinen rakenne), BatchNormalization ja ReLU kummankin jälkeen, MaxPooling ja Dropout, joka kasvaa syvemmälle mentäessä (0.2 → 0.3 → 0.4).
- Kaksi peräkkäistä 3×3-konvoluutiota näkevät yhdessä 5×5-alueen, mutta niissä on vähemmän parametreja ja yksi epälineaarisuus enemmän kuin yhdessä 5×5-kerroksessa.
- `padding="same"` säilyttää kuvan koon konvoluutiossa, joten koko pienenee vain poolauksessa: 32 → 16 → 8 → 4.
- Suodattimien määrä kaksinkertaistuu, kun spatiaalinen koko puolittuu: syvemmät kerrokset oppivat monimutkaisempia ja abstraktimpia piirteitä.
- Lopussa käytetään **GlobalAveragePooling2D**-kerrosta Flattenin sijaan. Se laskee jokaisen piirrekartan keskiarvon, mikä vähentää parametreja merkittävästi ja ylisovittumista.

**Parannettu CNN + augmentaatio (`cnn_aug`)**: täsmälleen sama arkkitehtuuri, mutta ensimmäisenä kerroksena on augmentaatio. Augmentaatiokerroksella ei ole opetettavia parametreja.

Alla on augmentoidun mallin rakenne ja parametrimäärä.

In [ ]:
def cnn_baseline():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="cnn_baseline")


def conv_block(filters, dropout):
    return [layers.Conv2D(filters, 3, padding="same", use_bias=False),
            layers.BatchNormalization(), layers.Activation("relu"),
            layers.Conv2D(filters, 3, padding="same", use_bias=False),
            layers.BatchNormalization(), layers.Activation("relu"),
            layers.MaxPooling2D(),
            layers.Dropout(dropout)]


def cnn_improved(use_augment):
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        *([augment] if use_augment else []),
        *conv_block(32, 0.2),
        *conv_block(64, 0.3),
        *conv_block(128, 0.4),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(10, activation="softmax"),
    ], name="cnn_aug" if use_augment else "cnn_bn_dropout")


cnn_improved(True).summary()

## Koulutus

Kaikki mallit koulutetaan samalla `train`-funktiolla, jotta vertailu on reilu: Adam-optimoija (oppimisnopeus 0.001), häviöfunktiona `sparse_categorical_crossentropy` (luokat ovat kokonaislukuja 0–9), eräkoko 128 ja 10 % opetusdatasta validaatioon.

Kaksi callbackia ohjaa koulutusta:
- **EarlyStopping** lopettaa koulutuksen, kun validaatiohäviö ei ole parantunut `PATIENCE` epookkiin, ja palauttaa parhaan epookin painot. Näin epookkien enimmäismäärä (60) toimii vain ylärajana, eikä ylisovittunutta mallia päädytä käyttämään.
- **ReduceLROnPlateau** puolittaa oppimisnopeuden, jos validaatiohäviö jumittaa kolme epookkia. Pienempi askel auttaa hienosäätämään painoja koulutuksen loppuvaiheessa.

Opetusaika mitataan `time.perf_counter()`-funktiolla `fit`-kutsun ympäriltä. Aika riippuu laitteistosta (CPU/GPU), joten se on vertailukelpoinen vain saman koneen ajojen kesken.

In [ ]:
def train(model, x, y, epochs, batch_size=128, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ]
    start = time.perf_counter()
    history = model.fit(x, y, validation_split=0.1, epochs=epochs,
                        batch_size=batch_size, callbacks=callbacks, verbose=2)
    duration = time.perf_counter() - start
    return history, duration

In [ ]:
baseline = cnn_baseline()
h_base, t_base = train(baseline, x_train_n, y_train, EPOCHS)

In [ ]:
bn_model = cnn_improved(use_augment=False)
h_bn, t_bn = train(bn_model, x_train_n, y_train, EPOCHS)

In [ ]:
aug_model = cnn_improved(use_augment=True)
h_aug, t_aug = train(aug_model, x_train_n, y_train, EPOCHS)

## Oppimiskäyrät

Kuvaajissa yhtenäinen viiva on opetusdata ja katkoviiva validaatiodata. Olennaista on näiden välinen ero: jos opetushäviö laskee mutta validaatiohäviö kääntyy nousuun, malli ylisovittuu eli opettelee opetuskuvien yksityiskohtia yleisten piirteiden sijaan.

Perusmallissa ilman regularisointia ero kasvaa nopeasti, ja validaatiohäviö alkaa nousta jo muutaman epookin jälkeen, minkä vuoksi early stopping pysäyttää sen aikaisin. Regularisoidussa mallissa käyrät pysyvät lähempänä toisiaan ja validaatiotarkkuus jatkaa nousua pidempään. Kohdissa, joissa oppimisnopeus puolittuu, käyrissä näkyy usein pieni porras alaspäin (häviö) tai ylöspäin (tarkkuus).

Augmentaatiomallissa opetustarkkuus voi jäädä validaatiotarkkuuden alle, koska opetuskuvat ovat muunneltuja ja siten vaikeampia ja koska dropout on päällä vain koulutuksessa. Tämä on merkki siitä, ettei malli ylisovitu.

In [ ]:
HISTORIES = {'perus': h_base, 'BN+dropout': h_bn, '+augmentaatio': h_aug}
def plot_history(histories):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax[0].plot(h.history["loss"], label=f"{name} train")
        ax[0].plot(h.history["val_loss"], "--", label=f"{name} val")
        ax[1].plot(h.history["accuracy"], label=f"{name} train")
        ax[1].plot(h.history["val_accuracy"], "--", label=f"{name} val")
    ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
    for a in ax:
        a.set_xlabel("epookki"); a.legend(fontsize=8); a.grid(alpha=0.3)
    plt.show()

plot_history(HISTORIES)

## Testitulokset

Taulukko näyttää kunkin CNN-mallin testitarkkuuden, parametrien määrän, opetusajan ja epookkien määrän. Testidataa käytetään vain tässä lopullisessa arvioinnissa.

Jokainen parannus nostaa tarkkuutta: syvempi verkko regularisoinnilla parantaa perusmallista selvästi, ja augmentaatio parantaa vielä lisää, koska se vähentää ylisovittumista ja opettaa sijainnin ja suunnan muutoksia sietäviä piirteitä. Augmentaatiomallin koulutus kestää yleensä pisimpään, koska se oppii pidempään ennen kuin early stopping pysäyttää sen. Alla olevassa solussa tuloksia verrataan tehtävänannon tavoitteisiin.

Parasta mallia (augmentaatio) tarkastellaan sen jälkeen luokittain. Kuten FCN:ssä, ajoneuvot ja laiva tunnistetaan parhaiten ja kissa on vaikein luokka; suurin sekaannus on kissan ja koiran välillä. Ero FCN:ään on kuitenkin selvä: diagonaali on huomattavasti vahvempi kaikissa luokissa.

In [ ]:
X_TEST = x_test_n
MODELS = {'cnn_baseline': (baseline, h_base, t_base),
          'cnn_bn_dropout': (bn_model, h_bn, t_bn),
          'cnn_aug': (aug_model, h_aug, t_aug)}
results = {}
for name, (m, h, t) in MODELS.items():
    loss, acc = m.evaluate(X_TEST, y_test, verbose=0)
    results[name] = {"test_acc": float(acc), "params": int(m.count_params()),
                     "train_time_s": round(t, 1), "epochs": len(h.history["loss"])}
    print(f"{name:12s} tarkkuus {acc:.4f} | parametrit {m.count_params():,} | "
          f"aika {t:.0f} s | epookit {len(h.history['loss'])}")
best_model = aug_model

In [ ]:
# verrataan tuloksia tehtävänannon tavoitteisiin
for name, r in results.items():
    acc = r["test_acc"] * 100
    taso = ("kunnianhimoinen tavoite (82-85 %) saavutettu" if acc >= 82 else
            "realistinen tavoite (75-80 %) saavutettu" if acc >= 75 else "alle tavoitteen")
    print(f"{name:16s} {acc:5.1f} %  -> {taso}")

In [ ]:
probs = best_model.predict(X_TEST, verbose=0)
y_pred = probs.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize="true"),
                       display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, values_format=".2f", cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (rivit normalisoitu)")
plt.show()

## Todennäköisyysjakaumat

Softmax-kerros antaa jokaiselle kuvalle todennäköisyysjakauman kymmenen luokan yli. Palkkikuvioissa vihreä on oikea luokka ja punainen mallin ennuste, jos se on väärä.

Oikein luokitelluissa kuvissa todennäköisyys on usein keskittynyt lähes kokonaan yhteen luokkaan. Histogrammi näyttää mallin varmuuden (suurimman todennäköisyyden) jakauman oikein ja väärin luokitelluille kuville erikseen: oikeat ennusteet painottuvat lähelle 100 %:a, kun taas väärien ennusteiden varmuus on keskimäärin selvästi pienempi. Malli siis jossain määrin ”tietää” milloin se on epävarma, mikä on hyödyllistä: matalan varmuuden ennusteet voisi käytännön sovelluksessa ohjata tarkistettavaksi. Kaikkiin ennusteisiin ei silti voi luottaa, sillä osa virheistä tehdään suurella varmuudella.

In [ ]:
def show_with_probs(indices, title):
    fig, ax = plt.subplots(len(indices), 2, figsize=(8, 2.2 * len(indices)),
                           gridspec_kw={"width_ratios": [1, 3]})
    for row, i in enumerate(indices):
        ax[row, 0].imshow(x_test[i]); ax[row, 0].axis("off")
        ax[row, 0].set_title(f"oikea: {class_names[y_test[i]]}", fontsize=8)
        colors = ["tab:green" if c == y_test[i] else
                  ("tab:red" if c == y_pred[i] else "tab:gray") for c in range(10)]
        ax[row, 1].barh(class_names, probs[i], color=colors)
        ax[row, 1].set_xlim(0, 1); ax[row, 1].invert_yaxis()
        ax[row, 1].tick_params(labelsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

correct = np.where(y_pred == y_test)[0]
show_with_probs(correct[:4], "Oikein luokitellut (vihreä = oikea luokka)")

In [ ]:
conf = probs.max(axis=1)
plt.figure(figsize=(7, 3.5))
plt.hist(conf[y_pred == y_test], bins=30, alpha=0.7, label="oikein")
plt.hist(conf[y_pred != y_test], bins=30, alpha=0.7, label="väärin")
plt.xlabel("mallin varmuus (suurin todennäköisyys)"); plt.ylabel("kuvia")
plt.legend(); plt.show()
print(f"keskimääräinen varmuus: oikein {conf[y_pred == y_test].mean():.2f}, "
      f"väärin {conf[y_pred != y_test].mean():.2f}")

## Väärin luokitellut kuvat

Virheet on järjestetty mallin varmuuden mukaan, joten ensimmäisinä näkyvät tapaukset, joissa malli on ollut varmimmin väärässä. Tyypillisesti virheet osuvat samankaltaisten luokkien välille: kissa ↔ koira, auto ↔ kuorma-auto, peura ↔ hevonen ja lintu ↔ lentokone. Myös sekaannusmatriisin suurimmat diagonaalin ulkopuoliset arvot sijoittuvat näihin pareihin.

Osa virheistä on ymmärrettäviä myös ihmisen silmin: kohde on hyvin pieni, osittain rajattu pois kuvasta, epätavallisesta kulmasta kuvattu tai taustaltaan harhaanjohtava (esim. lintu sinistä taivasta vasten näyttää lentokoneelta, ja vesistö taustalla vetää kohti laiva-luokkaa). Muutama testikuva on myös tulkinnanvarainen. Tällaisia virheitä on vaikea korjata pelkällä arkkitehtuurilla 32×32-resoluutiolla.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
# varmimmat virheet ensin
wrong = wrong[np.argsort(-conf[wrong])]

plt.figure(figsize=(12, 6))
for k, i in enumerate(wrong[:15]):
    plt.subplot(3, 5, k + 1)
    plt.imshow(x_test[i]); plt.axis("off")
    plt.title(f"oikea: {class_names[y_test[i]]}\nennuste: {class_names[y_pred[i]]} "
              f"({conf[i]:.0%})", fontsize=8)
plt.suptitle("Varmimmat väärät ennusteet")
plt.tight_layout(); plt.show()

show_with_probs(wrong[:4], "Väärin luokitellut (punainen = ennuste, vihreä = oikea)")

## Mitä konvoluutiokerros oppii

Ensimmäisen konvoluutiokerroksen 32 suodatinta ovat 3×3×3-painomatriiseja, jotka voidaan piirtää pieninä RGB-kuvina (arvot skaalattu välille 0–1). Jokainen suodatin liukuu koko kuvan yli ja reagoi tiettyyn paikalliseen kuvioon.

Suodattimissa näkyy tyypillisesti kahdenlaisia rakenteita: **reuna- ja viivatunnistimia**, joissa vaalea ja tumma puoli vaihtelevat eri suunnissa (pysty, vaaka, vino), sekä **väritunnistimia**, jotka reagoivat tiettyyn väriin tai värien kontrastiin. 3×3-koko on niin pieni, että kuviot ovat karkeita, mutta periaate on sama kuin isommissa verkoissa. Syvemmät kerrokset yhdistävät näitä yksinkertaisia piirteitä yhä monimutkaisemmiksi, kuten kulmiksi, tekstuureiksi ja lopulta kohteen osiksi (silmät, pyörät, siivet).

FCN:ssä vastaavaa ei synny, koska sen ensimmäisen kerroksen jokainen neuroni katsoo koko kuvaa kerralla eikä samaa piirrettä hyödynnetä kuvan eri kohdissa.

In [ ]:
w = [l for l in aug_model.layers if isinstance(l, layers.Conv2D)][0].get_weights()[0]
w = (w - w.min()) / (w.max() - w.min())
plt.figure(figsize=(8, 4))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(w[:, :, :, i]); plt.axis("off")
plt.suptitle("Ensimmäisen Conv2D-kerroksen 3×3-suodattimet (RGB)"); plt.show()

## FCN vs CNN

Pylväskaaviot ja taulukko vertailevat kaikkia viittä mallia neljällä mittarilla.

- **Testitarkkuus:** jo yksinkertainen perus-CNN, jossa on vain kaksi konvoluutiokerrosta, on selvästi parempi kuin paras FCN. Parannettu CNN augmentaatiolla vie eron vielä suuremmaksi.
- **Parametrit:** paras FCN sisältää useita miljoonia parametreja, kun taas paras CNN pärjää murto-osalla. CNN on siis sekä tarkempi että pienempi – parametrien määrä ei ratkaise, vaan se, miten ne käytetään.
- **Opetusaika:** konvoluutio vaatii enemmän laskentaa parametria kohden kuin tiheä kerros, joten CNN:n epookki on hitaampi ja augmentaatio pidentää koulutusta entisestään. Aika riippuu vahvasti siitä, onko käytössä GPU.
- **Oppimiskäyrät:** alla olevassa kuvaajassa FCN:n validaatiohäviö tasaantuu ja validaatiotarkkuus jää selvästi alemmalle tasolle, kun taas CNN:n validaatiotarkkuus jatkaa nousua pidempään ja asettuu korkeammalle. FCN:n opetus- ja validaatiokäyrien välinen ero on myös suurempi eli se ylisovittuu herkemmin.

In [ ]:
with open("results_fcn.json") as f:
    fcn_res = json.load(f)
all_res = {**{k: v for k, v in fcn_res.items()}, **results}

names = list(all_res)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].bar(names, [all_res[n]["test_acc"] for n in names]); ax[0].set_title("Testitarkkuus")
ax[1].bar(names, [all_res[n]["params"] for n in names]); ax[1].set_title("Parametrit")
ax[2].bar(names, [all_res[n]["train_time_s"] for n in names]); ax[2].set_title("Opetusaika (s)")
for a in ax:
    a.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()

print(f"{'malli':16s}{'tarkkuus':>10s}{'parametrit':>13s}{'aika (s)':>10s}{'epookit':>9s}")
for n in names:
    r = all_res[n]
    print(f"{n:16s}{r['test_acc']:>10.4f}{r['params']:>13,}{r['train_time_s']:>10.0f}{r['epochs']:>9d}")

In [ ]:
fh = fcn_res["fcn_improved"]["history"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(fh["loss"], label="FCN train"); ax[0].plot(fh["val_loss"], "--", label="FCN val")
ax[0].plot(h_aug.history["loss"], label="CNN train"); ax[0].plot(h_aug.history["val_loss"], "--", label="CNN val")
ax[1].plot(fh["val_accuracy"], label="FCN val"); ax[1].plot(h_aug.history["val_accuracy"], label="CNN val")
ax[0].set_title("Loss"); ax[1].set_title("Validaatiotarkkuus")
for a in ax:
    a.set_xlabel("epookki"); a.legend(); a.grid(alpha=0.3)
plt.show()

## Miksi CNN toimii paremmin kuvadatalle

CNN:n etu perustuu siihen, että sen rakenne vastaa kuvadatan ominaisuuksia:

1. **Paikallisuus.** Kuvan merkitykselliset piirteet (reunat, kulmat, tekstuurit) muodostuvat vierekkäisistä pikseleistä. Konvoluutio katsoo kerralla pientä aluetta, kun taas FCN litistää kuvan vektoriksi ja hukkaa tiedon siitä, mitkä pikselit ovat vierekkäin. FCN:lle pikselien järjestyksellä ei ole merkitystä – sen pitäisi oppia kuvan 2D-rakenne itse datasta.
2. **Painojen jakaminen.** Sama suodatin liukuu koko kuvan yli, joten reunantunnistin opitaan kerran ja sitä käytetään kaikkialla. Tämä vähentää parametreja dramaattisesti ja estää ylisovittumista. FCN joutuu oppimaan saman piirteen erikseen jokaiseen kuvan kohtaan.
3. **Siirtosietoisuus.** Konvoluutio ja poolaus tekevät mallista sietokykyisen kohteen sijainnin muutoksille: kissa vasemmassa reunassa ja oikeassa reunassa tuottaa samanlaiset piirteet. FCN:lle sama kohde eri kohdassa on täysin eri syöte.
4. **Hierarkkiset piirteet.** Peräkkäiset konvoluutiokerrokset yhdistävät yksinkertaisia piirteitä monimutkaisemmiksi (reunat → muodot → kohteen osat), mikä vastaa kuvien luonnollista rakennetta.
5. **Värikanavat yhdessä.** Konvoluutiosuodatin käsittelee samassa kohdassa kaikki kolme värikanavaa, joten se oppii sekä muoto- että väripiirteitä samanaikaisesti.

Tämän ansiosta CNN on yhtä aikaa tarkempi, pienempi ja yleistää paremmin, mikä näkyy yllä olevassa vertailussa.

## Kokeilut ja johtopäätökset

**Kokeillut asiat**
- *Esikäsittely värikuville:* kanavakohtainen standardointi opetusdatan tilastoilla (sama molemmissa notebookeissa).
- *Arkkitehtuuri:* MNIST-tyylisestä kahden konvoluutiokerroksen verkosta siirryttiin kolmen lohkon VGG-tyyliseen verkkoon, jossa on kaksi konvoluutiota lohkoa kohden ja jossa suodattimien määrä kasvaa (32 → 64 → 128). Flatten korvattiin GlobalAveragePooling-kerroksella.
- *Regularisointi:* Batch Normalization jokaisen konvoluution jälkeen ja syvemmälle kasvava Dropout.
- *Data-augmentaatio:* vaakapeilaus ja satunnainen siirto.
- *Koulutus:* early stopping ja oppimisnopeuden puolitus.

**Johtopäätökset**
- CNN on selvästi FCN:ää parempi CIFAR-10-datalle kaikilla mittareilla opetusaikaa lukuun ottamatta, ja se saavuttaa paremman tuloksen pienemmällä parametrimäärällä.
- Mallien kehitys osoittaa, että jokaisella parannuksella on oma vaikutuksensa: syvyys ja BatchNorm lisäävät mallin kapasiteettia ja vakauttavat koulutusta, Dropout ja augmentaatio vähentävät ylisovittumista. Suurimmat yksittäiset hyödyt tulevat siirtymisestä FCN:stä konvoluutioon sekä augmentaatiosta.
- Molemmissa verkoissa vaikeimmat luokat ovat samat (kissa, koira, lintu, peura), mutta CNN tekee niissäkin huomattavasti vähemmän virheitä. Jäljelle jääneet virheet ovat usein aidosti vaikeita tapauksia, joissa kuva on epäselvä tai kohde pieni.
- Mallin varmuus kertoo jotain sen luotettavuudesta: väärät ennusteet ovat keskimäärin epävarmempia kuin oikeat.

**Jatkokehitysideoita**
Tarkkuutta voisi parantaa edelleen esimerkiksi syvemmällä verkolla tai residuaaliyhteyksillä (ResNet), voimakkaammalla augmentaatiolla (Cutout, Mixup, satunnainen rajaus), oppimisnopeuden aikataulutuksella (cosine decay, one-cycle), L2-regularisoinnilla sekä usean mallin ensemblella. Myös siirto-oppiminen esikoulutetusta verkosta toisi todennäköisesti selvän parannuksen.